# Red Wine Quality Prediction — Google Colab

Predict a wine's quality score from 11 laboratory measurements. This is **regression**: decimal predictions are valid.

This notebook covers data exploration, training/testing, and saving the fitted model and parameters. Run it in **Google Colab or standalone Jupyter**. Later, build FastAPI and Docker in VS Code.

Dataset: [Kaggle Red Wine Quality](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009), file **winequality-red.csv**.
Original source: Cortez et al. (2009), [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality).

This notebook is intentionally unexecuted. Its outputs must come from your uploaded CSV and your Colab run.

## Before running

1. Open [Google Colab](https://colab.research.google.com/), choose **Upload notebook**, and upload this .ipynb.
2. Save a copy in Google Drive if needed, named **wine_quality_training.ipynb**.
3. Use a Python CPU runtime; no GPU is needed.
4. Run each code cell in order with **Shift+Enter**, waiting for it to finish.
5. When asked for data, upload your **CSV**. Extract a Kaggle ZIP on your computer first.

Colab normally includes the libraries below. If an import fails, run the following in a separate code cell, restart the runtime, then run from the top:

    %pip install pandas numpy scipy scikit-learn matplotlib joblib

Do not change packages halfway through training.


## 1. Import libraries and record the environment

A fixed seed makes our random choices repeatable. We will export exact package versions for the later API and Docker environment.


In [ ]:
import io
import json
import platform
import shutil
from importlib.metadata import version
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
RUNTIME_PACKAGES = ["numpy", "scipy", "pandas", "scikit-learn", "joblib", "threadpoolctl"]
package_versions = {name: version(name) for name in RUNTIME_PACKAGES}
print("Python:", platform.python_version())
display(pd.Series(package_versions, name="Installed version").to_frame())


## 2. Upload and read your red-wine CSV

Choose your CSV when Colab prompts you. The reader automatically detects comma or semicolon separators.
In standalone Jupyter, place the CSV beside the notebook and adjust DATA_PATH if its name differs.


In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("Upload exactly one red-wine CSV. Extract any ZIP first.")
    dataset_name = csv_names[0]
    raw_bytes = uploaded[dataset_name]
else:
    DATA_PATH = Path("winequality-red.csv")
    dataset_name = DATA_PATH.name
    raw_bytes = DATA_PATH.read_bytes()

df = pd.read_csv(io.BytesIO(raw_bytes), sep=None, engine="python")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
print("Dataset:", dataset_name)
print("Original shape:", df.shape)
display(df.head())


## 3. Validate and clean the dataset

Column names are standardized for the future API: fixed acidity becomes fixed_acidity, and pH becomes ph.
An optional CSV index/ID is removed. Unexpected columns trigger an error so combined red/white data is not silently accepted.

Remove exact duplicate complete records and report the count. Differing labels for identical measurements remain and will be grouped.
Missing inputs, if any, will be filled using training-only medians inside the pipeline. Missing target labels are excluded.


In [ ]:
FEATURES = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide",
    "density", "ph", "sulphates", "alcohol",
]
TARGET = "quality"
index_columns = [c for c in df.columns if c.startswith("unnamed:") or c == "id"]
df = df.drop(columns=index_columns)
required = FEATURES + [TARGET]
missing_columns = sorted(set(required) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(required))
if missing_columns or extra_columns or not df.columns.is_unique:
    raise ValueError(f"Check the red-wine CSV. Missing: {missing_columns}; unexpected: {extra_columns}.")

df = df.loc[:, required].apply(pd.to_numeric, errors="raise")
if np.isinf(df.to_numpy(dtype=float)).any():
    raise ValueError("Infinity found. Inspect the CSV before continuing.")
missing_labels = int(df[TARGET].isna().sum())
df = df.dropna(subset=[TARGET])
if not df[TARGET].between(0, 10).all():
    raise ValueError("Quality must be between 0 and 10.")
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
if len(df) < 100 or df[FEATURES].isna().all().any():
    raise ValueError("Too few rows or an entirely missing input column. Check the CSV.")

print("Rows with missing target removed:", missing_labels)
print("Exact duplicate rows removed:", duplicates_removed)
print("Cleaned shape:", df.shape)
display(df.isna().sum().rename("Missing values").to_frame())


## 4. Reserve the test set

X contains the 11 inputs; y contains quality. Identical input measurements share a group and stay on the same side of every split.
The split is approximately 80/20 because groups, rather than individual rows, are split.

**Do not use the test set to choose models or tune settings.**


In [ ]:
X, y = df[FEATURES].copy(), df[TARGET].copy()
groups = pd.factorize(pd.MultiIndex.from_frame(X), sort=False)[0]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups[train_idx], groups[test_idx]
assert set(groups_train).isdisjoint(set(groups_test))
if X_train.isna().all().any():
    raise ValueError("A training input is entirely missing. Inspect the split.")
print(f"Training: {len(X_train)} rows ({len(X_train) / len(X):.1%})")
print(f"Testing:  {len(X_test)} rows ({len(X_test) / len(X):.1%})")
print("Passed: identical inputs do not cross the train/test split.")


## 5. Explore the training data

Look for common quality ratings, differences in input scales, and associations with quality.
Correlation does not establish causation. Keep all 11 inputs for this first project. The test set stays reserved.


In [ ]:
train_data = X_train.assign(quality=y_train)
display(train_data.describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
y_train.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="firebrick")
axes[0].set(title="Training quality distribution", xlabel="Quality", ylabel="Number of samples")
axes[0].tick_params(axis="x", rotation=0)
train_data.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue"
)
axes[1].set(title="Training correlations with quality", xlabel="Pearson correlation")
plt.tight_layout()
plt.show()


## 6. Compare a simple baseline and Linear Regression

The median DummyRegressor always guesses the training median. A learned model should improve on this simple reference.
Linear Regression learns a weighted combination of the inputs.

A pipeline keeps imputation, scaling, and the model together. Preprocessing is fitted separately within every training fold to prevent leakage.
Scaling helps Linear Regression. Random Forest does not require it, but we retain a consistent pipeline interface.

Use **five-fold grouped cross-validation MAE** to choose a model. Lower MAE is better. Validation folds come entirely from the training set.


In [ ]:
def make_pipeline(regressor):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", regressor),
    ])

cv = GroupKFold(n_splits=5)
candidates = {
    "Median baseline": make_pipeline(DummyRegressor(strategy="median")),
    "Linear Regression": make_pipeline(LinearRegression()),
}
cv_rows = []
for name, pipeline in candidates.items():
    fold_mae = -cross_val_score(
        pipeline, X_train, y_train, groups=groups_train, cv=cv,
        scoring="neg_mean_absolute_error", n_jobs=-1,
    )
    cv_rows.append({
        "model": name, "cv_mae": float(fold_mae.mean()), "cv_std": float(fold_mae.std())
    })
display(pd.DataFrame(cv_rows).sort_values("cv_mae").round(4))


## 7. Tune Random Forest and choose the model

Random Forest averages many decision trees and can learn nonlinear patterns.
Try 8 combinations of tree depth, minimum leaf size, and features considered at each split, using the same five validation folds.
Wait for this longer-running cell to finish. The search parallelizes fits; each fit uses one worker.

Choose the learned model with the lowest training cross-validation MAE. The median baseline remains a reference.


In [ ]:
forest = make_pipeline(RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1))
search = RandomizedSearchCV(
    forest,
    param_distributions={
        "regressor__max_depth": [None, 8, 16],
        "regressor__min_samples_leaf": [1, 2, 4],
        "regressor__max_features": [0.7, 1.0],
    },
    n_iter=8, scoring="neg_mean_absolute_error", cv=cv,
    random_state=SEED, n_jobs=-1, refit=True, error_score="raise", verbose=1,
)
search.fit(X_train, y_train, groups=groups_train)
candidates["Tuned Random Forest"] = search.best_estimator_
cv_rows.append({
    "model": "Tuned Random Forest", "cv_mae": float(-search.best_score_),
    "cv_std": float(search.cv_results_["std_test_score"][search.best_index_]),
})
comparison = pd.DataFrame(cv_rows).sort_values("cv_mae").reset_index(drop=True)
print("Best forest settings:", search.best_params_)
display(comparison.round(4))

learned = comparison[comparison["model"] != "Median baseline"]
best_name = learned.iloc[0]["model"]
best_model = candidates[best_name]
best_model.fit(X_train, y_train)
baseline_model = candidates["Median baseline"].fit(X_train, y_train)
print("Chosen using training cross-validation:", best_name)
baseline_cv = comparison.loc[comparison["model"] == "Median baseline", "cv_mae"].iloc[0]
if learned.iloc[0]["cv_mae"] >= baseline_cv:
    print("Note: the learned model did not improve on the baseline in CV. Report this honestly.")


## 8. Evaluate once on the held-out test set

The winner was chosen using training cross-validation. Now evaluate it on unseen test rows. Do not tune settings or switch models using these results.

- **MAE:** average absolute error in quality-score points. For example, 0.5 would mean half a point of absolute error on average. This is an illustration, not a promised result.
- **RMSE:** error in quality-score points, penalizing larger errors more heavily.
- **R²:** 1 is perfect; 0 matches predicting the test-set mean; negative values are possible.

The baseline is included for context. Predictions can be decimals; do not round before computing metrics.


In [ ]:
def regression_metrics(actual, predicted):
    return {
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
        "r2": float(r2_score(actual, predicted)),
    }

test_predictions = best_model.predict(X_test)
test_metrics = regression_metrics(y_test, test_predictions)
baseline_metrics = regression_metrics(y_test, baseline_model.predict(X_test))
display(pd.DataFrame({best_name: test_metrics, "Median baseline": baseline_metrics}).T.round(4))

prediction_table = X_test.copy()
prediction_table["actual_quality"] = y_test
prediction_table["predicted_quality"] = test_predictions
prediction_table["absolute_error"] = np.abs(y_test.to_numpy() - test_predictions)
display(prediction_table[["actual_quality", "predicted_quality", "absolute_error"]].head(10).round(3))

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, test_predictions, alpha=0.45)
limits = [min(y_test.min(), test_predictions.min()) - 0.2, max(y_test.max(), test_predictions.max()) + 0.2]
ax.plot(limits, limits, "r--", label="Perfect prediction")
ax.set(xlim=limits, ylim=limits, xlabel="Actual quality", ylabel="Predicted quality", title=f"Held-out test: {best_name}")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Save the fitted model, parameters, and environment

**model.joblib** contains the fitted pipeline: imputation values, scaling statistics, and learned trees or regression coefficients.
**parameters.json** records readable hyperparameters; those settings alone cannot replace the fitted model.
**metadata.json** records feature order and exact package versions for the API.

We save the training-set fit that was evaluated, without refitting on test rows.
Later, match the Python minor version and runtime package pins in VS Code/Docker.


In [ ]:
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
joblib.dump(best_model, ARTIFACTS / "model.joblib", compress=3)

parameters = {
    "selected_model": best_name,
    "regressor_class": type(best_model.named_steps["regressor"]).__name__,
    "regressor_parameters": best_model.named_steps["regressor"].get_params(deep=False),
    "preprocessing": {"imputer_strategy": "median", "standard_scaler": True},
    "forest_search_best_parameters": search.best_params_,
    "random_seed": SEED,
}
metadata = {
    "task": "regression", "dataset": dataset_name,
    "dataset_source": "https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009",
    "target": TARGET, "features": FEATURES, "selected_model": best_name,
    "python_version": platform.python_version(),
    "python_major_minor": ".".join(platform.python_version().split(".")[:2]),
    "package_versions": package_versions,
    "training_rows": len(X_train), "test_rows": len(X_test),
    "duplicates_removed": duplicates_removed, "missing_target_rows_removed": missing_labels,
    "split": "80/20 GroupShuffleSplit; identical inputs kept together",
    "selection": "lowest five-fold grouped training CV MAE among learned models",
    "cv_results": comparison.to_dict(orient="records"),
}
metrics = {"selected_model": best_name, "test": test_metrics, "median_baseline_test": baseline_metrics}
sample_request = {
    name: (None if pd.isna(value) else float(value))
    for name, value in X_test.iloc[0].items()
}
for filename, payload in [
    ("parameters.json", parameters), ("metadata.json", metadata), ("metrics.json", metrics),
    ("sample_request.json", sample_request),
    ("expected_prediction.json", {"predicted_quality": float(test_predictions[0])}),
]:
    (ARTIFACTS / filename).write_text(
        json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8"
    )

requirements = "\n".join(f"{name}=={package_versions[name]}" for name in RUNTIME_PACKAGES) + "\n"
(ARTIFACTS / "requirements-model.txt").write_text(requirements, encoding="utf-8")
comparison.to_csv(ARTIFACTS / "cv_results.csv", index=False)
prediction_table.to_csv(ARTIFACTS / "test_predictions.csv", index=False)
print("Saved files:")
for path in sorted(ARTIFACTS.iterdir()):
    print(" -", path.name)
print("For the later Docker image, use Python", metadata["python_major_minor"])


## 10. Reload and verify the saved model

Check that the saved pipeline reproduces every held-out prediction. This is also the future FastAPI pattern: build a DataFrame in the saved feature order and call predict.


In [ ]:
loaded_model = joblib.load(ARTIFACTS / "model.joblib")
loaded_predictions = loaded_model.predict(X_test)
np.testing.assert_allclose(loaded_predictions, test_predictions, rtol=1e-10, atol=1e-10)

sample_frame = pd.DataFrame([sample_request], columns=FEATURES)
sample_prediction = float(loaded_model.predict(sample_frame)[0])
print("Reload check passed for every held-out test row.")
print("Example API input:")
print(json.dumps(sample_request, indent=2))
print("Predicted quality:", round(sample_prediction, 3))
print("Actual quality for this example:", float(y_test.iloc[0]))


## 11. Download artifacts and save the executed notebook

Download the ZIP and keep it for the VS Code stage. It includes the fitted model, settings, metrics, sample input/output, package pins, and test predictions.

After this cell finishes, select **File → Download → Download .ipynb** in Colab to save the executed notebook with outputs. Keep a Drive copy too.
Save screenshots of the cross-validation comparison, held-out metrics, and reload check as assignment evidence.

Colab runtime files are temporary: saving a notebook does not also save the separate artifact folder.


In [ ]:
zip_path = shutil.make_archive("wine_quality_artifacts", "zip", root_dir=ARTIFACTS)
if IN_COLAB:
    files.download(zip_path)
else:
    print("Artifacts ZIP:", Path(zip_path).resolve())
print("Training/testing and model saving are complete for this run.")
print("Next: build and test FastAPI in VS Code using these artifacts.")


## Explain your work in the assignment

Fill these in using your actual run:

- Problem: estimate red-wine quality from 11 physicochemical inputs.
- Preparation: removed __ identical complete records, grouped identical inputs, and fitted imputation/scaling inside training pipelines.
- Split: __ training rows and __ held-out test rows, random seed 42.
- Comparison: median baseline, Linear Regression, and tuned Random Forest.
- Selection: lowest five-fold grouped training CV MAE among learned models.
- Chosen model: __. Test MAE: __; RMSE: __; R²: __.
- Persistence: fitted pipeline saved as model.joblib, with hyperparameters and environment versions recorded separately.
- Verification: reloading reproduced all held-out predictions.
- Limitation: ratings are subjective and unevenly distributed. The metrics describe this dataset and split, not universal performance.

Keep the downloaded ZIP, the executed .ipynb, and the original CSV for the next stage.

References: [UCI source](https://archive.ics.uci.edu/dataset/186/wine+quality), [Kaggle download](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009), [grouped splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html), [model persistence](https://scikit-learn.org/stable/model_persistence.html).
